# LASTDANCE — EVA-CLIP dev-subset-5 gate

Notebook này chỉ chạy **dev gate EVA-CLIP** trên Kaggle C/Tesla T4. Nó gọi trực tiếp `python -m scripts.run_eva_clip_dev_gate`; không chứa và không tạo runner production 9 batch.

Gate chỉ PASS khi đủ 4.164 keyframe, dimension khớp output runtime thật, vector lưu `float16`, checkpoint đã interrupt/resume thật, validator PASS, `runtime.device=cuda` và `runtime.gpu_name=Tesla T4`. Mọi vector/checkpoint/manifest được ghi dưới `/kaggle/working`, không thuộc Git repo.


## Chuẩn bị trên Kaggle C

- Accelerator: **GPU T4**.
- Internet: **On** để clone code và tải đúng immutable revision từ Hugging Face.
- Gắn private Dataset **Eva test dataset** (`minhlight0204/eva-test-dataset`) chứa trực tiếp `frames.csv`, `frames.csv.state.json` và đúng 5 MP4 dev. Notebook ưu tiên `/kaggle/input/eva-test-dataset` nhưng tự dò path mount thực tế bằng đúng bộ file này, lọc catalog về 4.164 keyframe rồi decode mỗi MP4 đúng một lần; không quét production bundle và không giải nén TAR.
- Không gắn artifact EVA dev-gate cũ vào output path. Runner fail-closed nếu `/kaggle/working/visual-embeddings/dev-subset-5/eva_clip` đã tồn tại; khi đó hãy giữ session để điều tra hoặc khởi động session sạch.
- `HF_TOKEN` chỉ cần nếu chủ động bật cell upload bằng chứng cuối notebook. Không ghi hoặc in token trực tiếp.


## 1. Clone và khóa đúng commit chứa EVA runner


In [ ]:
from pathlib import Path
import os
import socket
import subprocess
import sys

WORKING_ROOT = Path("/kaggle/working")
REPO = WORKING_ROOT / "LASTDANCE"
BRANCH = "codex/offline-visual-embeddings"
EXPECTED_COMMIT = "63cdf244449e3eb8bcffd8d47a417ef2a07d7927"
CLONE_URL = "https://" + "github.com/ThanhVu165/LASTDANCE.git"

assert not any(marker in CLONE_URL for marker in "[]()"), CLONE_URL

try:
    socket.getaddrinfo("github.com", 443)
except socket.gaierror as error:
    raise RuntimeError(
        "Không phân giải được github.com. Hãy bật Internet trong "
        "Kaggle Notebook Settings rồi khởi động lại session."
    ) from error

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(
        f"{REPO} tồn tại nhưng không phải Git repo; hãy dùng session Kaggle sạch."
    )

if not (REPO / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            CLONE_URL,
            str(REPO),
        ],
        check=True,
    )

subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO, check=True)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)
assert (REPO / "scripts/run_eva_clip_dev_gate.py").is_file()

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("CODE CHECKOUT PASS:", actual_commit)


## 2. Cài dependency Kaggle GPU

Không cài đè Torch/Torchvision CUDA có sẵn trong image Kaggle.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/LASTDANCE")
assert (REPO / "requirements/kaggle-gpu.txt").is_file(), REPO

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "-r",
        "requirements/kaggle-gpu.txt",
    ],
    cwd=REPO,
    check=True,
)
print("DEPENDENCY INSTALL PASS")


## 3. Đọc thẳng Eva test dataset và trích keyframe

Cell này tự resolve **một** dataset dưới `/kaggle/input` có `frames.csv` + `frames.csv.state.json` và đúng năm MP4 `L21_V001`, `L21_V002`, `L21_V003`, `L21_V005`, `L21_V006`. Sau khi lọc đúng 4.164 record, FFmpeg decode tuần tự mỗi video đúng một lần để trích JPEG. Không chạy lại inventory, shot detection, keyframe planning, quality hoặc xử lý archive.


In [ ]:
import csv
import json
import os
import subprocess
import sys
from collections import defaultdict
from pathlib import Path
from tempfile import TemporaryDirectory

import torch

REPO = Path("/kaggle/working/LASTDANCE")
if not (REPO / "offline").is_dir():
    raise RuntimeError(
        f"Chưa có source repo tại {REPO}. Hãy chạy cell Clone/checkout trước."
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from offline.catalog import (
    FRAME_COLUMNS,
    validate_frames_catalog,
    write_frames_catalog_atomic,
)
from offline.visual_embeddings import load_embedding_catalog
from shared.schemas.frame import FrameRecord

STAGING_ROOT = Path("/kaggle/working/eva-clip-dev-input")
CATALOG = STAGING_ROOT / "catalog" / "frames.csv"
KEYFRAMES_ROOT = STAGING_ROOT / "keyframes"
EXPECTED_RECORD_COUNT = 4164
EXPECTED_GPU_NAME = "Tesla T4"
VIDEO_IDS = [
    "L21_V001",
    "L21_V002",
    "L21_V003",
    "L21_V005",
    "L21_V006",
]
VIDEO_ID_SET = set(VIDEO_IDS)

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
dataset_override = os.environ.get("EVA_TEST_DATASET_ROOT", "").strip()
if dataset_override:
    dataset_candidates = [Path(dataset_override).expanduser().resolve()]
else:
    preferred = KAGGLE_INPUT_ROOT / "eva-test-dataset"
    discovered = (
        [path.parent.resolve() for path in KAGGLE_INPUT_ROOT.rglob("frames.csv")]
        if KAGGLE_INPUT_ROOT.is_dir()
        else []
    )
    dataset_candidates = list(dict.fromkeys([preferred.resolve(), *discovered]))

matching_datasets = []
for candidate in dataset_candidates:
    if not (candidate / "frames.csv").is_file():
        continue
    if not (candidate / "frames.csv.state.json").is_file():
        continue
    mounted_video_ids = {
        path.stem
        for path in candidate.rglob("*.mp4")
        if path.is_file()
    }
    if mounted_video_ids == VIDEO_ID_SET:
        matching_datasets.append(candidate)

if len(matching_datasets) != 1:
    mounted_inputs = (
        sorted(str(path) for path in KAGGLE_INPUT_ROOT.iterdir())
        if KAGGLE_INPUT_ROOT.is_dir()
        else []
    )
    raise RuntimeError(
        "Không resolve được đúng một Eva test dataset. Trong Kaggle hãy chọn "
        "Add Input -> minhlight0204/eva-test-dataset rồi chạy lại cell. "
        f"Candidates hợp lệ={matching_datasets}; inputs đang mount={mounted_inputs}"
    )

DATASET_ROOT = matching_datasets[0]
SOURCE_CATALOG = DATASET_ROOT / "frames.csv"
SOURCE_CATALOG_STATE = DATASET_ROOT / "frames.csv.state.json"
assert SOURCE_CATALOG.is_file(), SOURCE_CATALOG
assert SOURCE_CATALOG_STATE.is_file(), SOURCE_CATALOG_STATE
assert validate_frames_catalog(SOURCE_CATALOG, SOURCE_CATALOG_STATE), (
    "frames.csv hoặc frames.csv.state.json không hợp lệ",
    DATASET_ROOT,
)

assert torch.cuda.is_available(), "CUDA unavailable; select a Kaggle GPU accelerator"
gpu_name = torch.cuda.get_device_name(0)
assert gpu_name == EXPECTED_GPU_NAME, (gpu_name, EXPECTED_GPU_NAME)

selected_records = []
with SOURCE_CATALOG.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == FRAME_COLUMNS, reader.fieldnames
    for row in reader:
        if row["video_id"] in VIDEO_ID_SET:
            selected_records.append(
                FrameRecord(**{**row, "window_id": row["window_id"] or None})
            )

assert len(selected_records) == EXPECTED_RECORD_COUNT, len(selected_records)
assert {record.video_id for record in selected_records} == VIDEO_ID_SET

source_catalog_state = json.loads(
    SOURCE_CATALOG_STATE.read_text(encoding="utf-8")
)
selected_sources = [
    source
    for source in source_catalog_state["sources"]
    if source["video_id"] in VIDEO_ID_SET
]
assert len(selected_sources) == len(VIDEO_IDS)

write_frames_catalog_atomic(
    CATALOG,
    records=selected_records,
    sources=selected_sources,
)
assert validate_frames_catalog(CATALOG), CATALOG

video_paths = {
    path.stem: path.resolve()
    for path in DATASET_ROOT.rglob("*.mp4")
    if path.is_file()
}
assert set(video_paths) == VIDEO_ID_SET, (
    "Dataset phải chứa đúng 5 MP4 dev",
    sorted(video_paths),
)

records_by_video = defaultdict(list)
for record in selected_records:
    records_by_video[record.video_id].append(record)

KEYFRAMES_ROOT.mkdir(parents=True, exist_ok=True)
for video_id in VIDEO_IDS:
    records = sorted(records_by_video[video_id], key=lambda row: row.frame_id)
    frame_ids = [record.frame_id for record in records]
    assert frame_ids == sorted(set(frame_ids)), video_id
    destinations = [
        KEYFRAMES_ROOT / video_id / f"{record.shot_id}_{record.local_idx}.jpg"
        for record in records
    ]
    if all(path.is_file() and path.stat().st_size > 0 for path in destinations):
        print("KEYFRAMES ALREADY READY:", video_id, len(destinations))
        continue

    with TemporaryDirectory(
        prefix=f".{video_id}-extract-",
        dir=KEYFRAMES_ROOT,
    ) as temporary_folder:
        temporary_root = Path(temporary_folder)
        output_pattern = temporary_root / "%08d.jpg"
        selection = "+".join(f"eq(n\\,{frame_id})" for frame_id in frame_ids)
        command = [
            "ffmpeg",
            "-nostdin",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            str(video_paths[video_id]),
            "-vf",
            f"select={selection}",
            "-vsync",
            "vfr",
            "-frames:v",
            str(len(records)),
            "-start_number",
            "0",
            "-q:v",
            "3",
            "-y",
            str(output_pattern),
        ]
        print("EXTRACTING ONCE:", video_id, len(records), "frames")
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
        )
        assert completed.returncode == 0, completed.stderr[-4000:]
        staged = [
            temporary_root / f"{index:08d}.jpg"
            for index in range(len(records))
        ]
        assert all(path.is_file() and path.stat().st_size > 0 for path in staged), (
            video_id,
            len(list(temporary_root.glob("*.jpg"))),
            len(records),
        )
        for source, destination in zip(staged, destinations, strict=True):
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_name(destination.stem + ".tmp.jpg")
            source.replace(temporary)
            temporary.replace(destination)

selected_items = load_embedding_catalog(
    CATALOG,
    keyframes_root=KEYFRAMES_ROOT,
    video_ids=VIDEO_ID_SET,
)
assert len(selected_items) == EXPECTED_RECORD_COUNT

VIDEO_ID_FILE = Path("/kaggle/working/worker-dev-subset-5.txt")
VIDEO_ID_FILE.write_text("\n".join(VIDEO_IDS) + "\n", encoding="utf-8")
EMBEDDING_ROOT = Path("/kaggle/working/visual-embeddings")
ARTIFACT = EMBEDDING_ROOT / "dev-subset-5" / "eva_clip"
assert not ARTIFACT.exists(), (
    f"Artifact đã tồn tại: {ARTIFACT}. Không ghi đè; hãy điều tra hoặc dùng session sạch."
)

os.environ["CATALOG"] = str(CATALOG)
os.environ["KEYFRAMES_ROOT"] = str(KEYFRAMES_ROOT)
os.environ["EMBEDDING_ROOT"] = str(EMBEDDING_ROOT)
os.environ["HF_HOME"] = "/kaggle/working/huggingface-cache"

subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.environment_doctor",
        "--profile",
        "kaggle-gpu",
        "--skip-data",
    ],
    cwd=REPO,
    check=True,
)
print("DEV INPUT PREFLIGHT PASS")
print("GPU:", gpu_name)
print("dataset_root:", DATASET_ROOT)
print("source_catalog:", SOURCE_CATALOG)
print("dev_catalog:", CATALOG)
print("keyframes_root:", KEYFRAMES_ROOT)
print("videos:", len(VIDEO_IDS), VIDEO_IDS)
print("records:", len(selected_items))
print("worker_file:", VIDEO_ID_FILE)


## 4. Chạy đúng EVA-CLIP dev-gate runner

Runner tự chạy verifier thật, intentional interruption với exit 75, kiểm checkpoint partial, mở process mới để resume, gọi validator, rồi chỉ in `EVA_CLIP_DEV_GATE_PASS` khi toàn bộ contract đạt. Batch size 32 là một phần của checkpoint signature.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/LASTDANCE")
CATALOG = Path(os.environ["CATALOG"])
KEYFRAMES_ROOT = Path(os.environ["KEYFRAMES_ROOT"])
EMBEDDING_ROOT = Path(os.environ["EMBEDDING_ROOT"])
VIDEO_ID_FILE = Path("/kaggle/working/worker-dev-subset-5.txt")

gate_command = [
    sys.executable,
    "-m",
    "scripts.run_eva_clip_dev_gate",
    "--catalog",
    str(CATALOG),
    "--keyframes-root",
    str(KEYFRAMES_ROOT),
    "--embedding-root",
    str(EMBEDDING_ROOT),
    "--video-id-file",
    str(VIDEO_ID_FILE),
    "--batch-id",
    "dev-subset-5",
    "--batch-size",
    "32",
    "--expected-record-count",
    "4164",
    "--expected-gpu-name",
    "Tesla T4",
]

print("RUNNING:", " ".join(gate_command))
subprocess.run(gate_command, cwd=REPO, check=True)


## 5. Xác nhận artifact thật sau gate

Dimension không được chốt chỉ từ con số ghi trong notebook: cell này đọc `vectors.npy` thật của mọi shard, lấy dimension quan sát được, rồi đối chiếu manifest và registry. Nó cũng kiểm đủ UID, `float16`, finite, L2 norm, checkpoint resume và runtime T4/CUDA.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np

REPO = Path("/kaggle/working/LASTDANCE")
if not (REPO / "offline").is_dir():
    raise RuntimeError(
        f"Chưa có source repo tại {REPO}. Hãy chạy cell Clone/checkout trước."
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from offline.visual_models import load_model_config

EXPECTED_COMMIT = "63cdf244449e3eb8bcffd8d47a417ef2a07d7927"
EXPECTED_MODEL_REVISION = "bf4190eb65dd5204ffb03e980108beb1200e0873"
EXPECTED_RECORD_COUNT = 4164
ARTIFACT = Path("/kaggle/working/visual-embeddings/dev-subset-5/eva_clip")

manifest = json.loads((ARTIFACT / "manifest.json").read_text(encoding="utf-8"))
checkpoint = json.loads((ARTIFACT / "checkpoint.json").read_text(encoding="utf-8"))
model_config = load_model_config("eva_clip")

shard_dirs = sorted(
    path for path in (ARTIFACT / "shards").iterdir()
    if path.is_dir() and path.name.isdigit()
)
assert shard_dirs, ARTIFACT

observed_dims = set()
observed_records = 0
observed_uids = []
norm_min = float("inf")
norm_max = 0.0

for shard_dir in shard_dirs:
    vectors = np.load(shard_dir / "vectors.npy", mmap_mode="r", allow_pickle=False)
    uids = np.load(shard_dir / "keyframe_uids.npy", mmap_mode="r", allow_pickle=False)
    assert vectors.dtype == np.float16, (shard_dir, vectors.dtype)
    assert vectors.ndim == 2 and vectors.shape[0] == len(uids), (shard_dir, vectors.shape)
    assert uids.dtype == np.int64 and uids.ndim == 1, (shard_dir, uids.dtype, uids.shape)
    assert np.isfinite(vectors).all(), shard_dir
    norms = np.linalg.norm(vectors.astype(np.float32), axis=1)
    assert np.allclose(norms, np.ones_like(norms), atol=5e-3, rtol=0), shard_dir
    observed_dims.add(int(vectors.shape[1]))
    observed_records += int(vectors.shape[0])
    observed_uids.extend(int(value) for value in uids)
    norm_min = min(norm_min, float(norms.min()))
    norm_max = max(norm_max, float(norms.max()))

assert len(observed_dims) == 1, observed_dims
observed_vector_dim = next(iter(observed_dims))
assert observed_records == EXPECTED_RECORD_COUNT
assert len(observed_uids) == len(set(observed_uids)) == EXPECTED_RECORD_COUNT

assert manifest["complete"] is True
assert manifest["modality"] == "eva_clip"
assert manifest["record_count"] == observed_records == EXPECTED_RECORD_COUNT
assert manifest["vector_dim"] == observed_vector_dim
assert observed_vector_dim == int(model_config["expected_vector_dim"])
assert manifest["vector_dtype"] == "float16"
assert manifest["checkpoint_resume_verified"] is True
assert manifest["model"]["id"] == model_config["model_id"]
assert manifest["model"]["revision"] == EXPECTED_MODEL_REVISION
assert manifest["runtime"]["device"] == "cuda"
assert manifest["runtime"]["gpu_name"] == "Tesla T4"
assert manifest["runtime"]["open_clip_torch"] == "3.3.0"
assert manifest["runtime"]["timm"] == "1.0.28"

assert checkpoint["complete"] is True
assert checkpoint["next_index"] == EXPECTED_RECORD_COUNT
assert checkpoint["intentional_interruption_observed"] is True
assert checkpoint["checkpoint_resume_verified"] is True

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert actual_commit == EXPECTED_COMMIT

gate_report = {
    "complete": True,
    "code_commit": actual_commit,
    "artifact": str(ARTIFACT),
    "record_count": observed_records,
    "vector_dim_runtime_observed": observed_vector_dim,
    "vector_dtype": manifest["vector_dtype"],
    "checkpoint_resume_verified": manifest["checkpoint_resume_verified"],
    "runtime": manifest["runtime"],
    "model": manifest["model"],
    "norm_min": norm_min,
    "norm_max": norm_max,
}

GATE_REPORT_PATH = Path("/kaggle/working/eva-clip-dev-gate-report.json")
GATE_REPORT_PATH.write_text(
    json.dumps(gate_report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("EVA-CLIP DEV-SUBSET-5 GATE VERIFIED")
print(json.dumps(gate_report, ensure_ascii=False, indent=2))
print("report:", GATE_REPORT_PATH)


## 6. Tùy chọn: archive và upload bằng chứng PASS lên HF Dataset

Mặc định `UPLOAD_GATE_EVIDENCE = False` để Run All không tự tạo remote state. Chỉ đổi thành `True` **sau khi cell gate verification PASS** và đã tạo Kaggle Secret `HF_TOKEN` có quyền Write. Archive chỉ chứa artifact dev gate, worker list và report; không chứa JPEG, model cache hoặc token. Remote namespace là `eva_clip/dev-gate/dev-subset-5/`, tách khỏi production.


In [ ]:
UPLOAD_GATE_EVIDENCE = False

if not UPLOAD_GATE_EVIDENCE:
    print("SKIP HF upload. Gate artifact remains under /kaggle/working.")
else:
    import hashlib
    import json
    import tarfile
    from pathlib import Path, PurePosixPath

    from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_REPO_ID = "MinhThuw0103/lastdance-visual-embeddings"
    HF_REPO_TYPE = "dataset"
    REMOTE_ROOT = "eva_clip/dev-gate/dev-subset-5"
    ARTIFACT = Path("/kaggle/working/visual-embeddings/dev-subset-5/eva_clip")
    VIDEO_ID_FILE = Path("/kaggle/working/worker-dev-subset-5.txt")
    GATE_REPORT_PATH = Path("/kaggle/working/eva-clip-dev-gate-report.json")
    ARCHIVE = Path("/kaggle/working/lastdance-eva-clip-dev-gate-63cdf24.tar.gz")
    CHECKSUM = ARCHIVE.with_suffix(ARCHIVE.suffix + ".sha256")

    assert ARTIFACT.is_dir()
    assert GATE_REPORT_PATH.is_file()
    gate_report = json.loads(GATE_REPORT_PATH.read_text(encoding="utf-8"))
    assert gate_report["complete"] is True
    assert gate_report["record_count"] == 4164
    assert gate_report["checkpoint_resume_verified"] is True
    assert gate_report["runtime"]["device"] == "cuda"
    assert gate_report["runtime"]["gpu_name"] == "Tesla T4"

    if not ARCHIVE.exists():
        with tarfile.open(ARCHIVE, "w:gz") as archive:
            archive.add(
                ARTIFACT,
                arcname="visual-embeddings/dev-subset-5/eva_clip",
            )
            archive.add(VIDEO_ID_FILE, arcname="workers/worker-dev-subset-5.txt")
            archive.add(GATE_REPORT_PATH, arcname="metadata/eva-clip-dev-gate-report.json")

    with tarfile.open(ARCHIVE, "r:gz") as archive:
        members = archive.getmembers()
        names = {member.name for member in members}
        for member in members:
            member_path = PurePosixPath(member.name)
            assert not member_path.is_absolute(), member.name
            assert ".." not in member_path.parts, member.name
            assert not member.issym() and not member.islnk(), member.name
        assert "visual-embeddings/dev-subset-5/eva_clip/manifest.json" in names
        assert "visual-embeddings/dev-subset-5/eva_clip/checkpoint.json" in names
        assert not any(
            PurePosixPath(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".mp4"}
            for name in names
        )

    digest = hashlib.sha256()
    with ARCHIVE.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    archive_sha256 = digest.hexdigest()
    CHECKSUM.write_text(f"{archive_sha256}  {ARCHIVE.name}\n", encoding="utf-8")

    token = UserSecretsClient().get_secret("HF_TOKEN")
    assert token and token.strip(), "Thiếu Kaggle Secret HF_TOKEN"
    api = HfApi(token=token)
    repo_info = api.repo_info(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
    assert repo_info.private is True, f"{HF_REPO_ID} phải là private Dataset"

    remote_archive = f"{REMOTE_ROOT}/{ARCHIVE.name}"
    remote_checksum = f"{REMOTE_ROOT}/{CHECKSUM.name}"
    remote_files = set(api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE))
    assert remote_archive not in remote_files and remote_checksum not in remote_files, (
        "Remote dev-gate evidence đã tồn tại; không tự ghi đè",
        remote_archive,
    )

    commit_info = api.create_commit(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        operations=[
            CommitOperationAdd(path_in_repo=remote_archive, path_or_fileobj=str(ARCHIVE)),
            CommitOperationAdd(path_in_repo=remote_checksum, path_or_fileobj=str(CHECKSUM)),
        ],
        commit_message="data(eva_clip): preserve dev-subset-5 gate evidence",
    )

    downloaded_checksum = Path(
        hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            filename=remote_checksum,
            token=token,
            force_download=True,
        )
    )
    remote_sha256 = downloaded_checksum.read_text(encoding="utf-8").split()[0].lower()
    assert remote_sha256 == archive_sha256, (remote_sha256, archive_sha256)

    print("EVA-CLIP DEV GATE EVIDENCE UPLOAD PASS")
    print("HF repo:", HF_REPO_ID)
    print("HF commit:", commit_info.oid)
    print("remote archive:", remote_archive)
    print("sha256:", archive_sha256)
